In [51]:
import tensorflow as tf
import os

print("="*50)
print("  MBTI Personality Classifier")
print("  NLP with Deep Learning (24-479-0212)")
print("="*50)

# Check CPU/GPU
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')

if gpus:
    print(f"✅ GPU detected: {gpus[0].name}")
else:
    print(f"✅ Running on CPU: {cpus[0].name}")
    print("   Estimated training time: ~20-30 minutes")
    print("   Make sure to save model after training!")

  MBTI Personality Classifier
  NLP with Deep Learning (24-479-0212)
✅ Running on CPU: /physical_device:CPU:0
   Estimated training time: ~20-30 minutes
   Make sure to save model after training!


In [52]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create all project folders
folders = [
    '/content/drive/MyDrive/MBTI_Classifier',
    '/content/drive/MyDrive/MBTI_Classifier/model',
    '/content/drive/MyDrive/MBTI_Classifier/assets',
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("✅ Google Drive mounted!")
print("📁 Folders created:")
for f in folders:
    print(f"   {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!
📁 Folders created:
   /content/drive/MyDrive/MBTI_Classifier
   /content/drive/MyDrive/MBTI_Classifier/model
   /content/drive/MyDrive/MBTI_Classifier/assets


In [53]:
MODEL_PATH     = "/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras"
TOKENIZER_PATH = "/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl"

model_exists     = os.path.exists(MODEL_PATH)
tokenizer_exists = os.path.exists(TOKENIZER_PATH)

print("Checking saved files...")
print(f"  Model:     {'✅ Found' if model_exists     else '❌ Not found'}")
print(f"  Tokenizer: {'✅ Found' if tokenizer_exists else '❌ Not found'}")

if model_exists and tokenizer_exists:
    print("\n⏩ Model already trained and saved!")
    print("   SKIP to CELL 10 to load it directly.")
    print("   No need to retrain!")
else:
    print("\n▶️  No saved model found.")
    print("   Continue with CELL 4 to train from scratch.")
    print("   Training takes ~20-30 mins on CPU.")

Checking saved files...
  Model:     ✅ Found
  Tokenizer: ✅ Found

⏩ Model already trained and saved!
   SKIP to CELL 10 to load it directly.
   No need to retrain!


In [54]:
print("Installing dependencies...")

!pip install -q tqdm
!pip install -q scikit-learn

import numpy as np
import pandas as pd
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, GlobalMaxPooling1D, BatchNormalization
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("✅ All dependencies installed and imported!")

Installing dependencies...
✅ All dependencies installed and imported!


In [55]:
import shutil, os

# ── Upload mbti_1.csv from Drive ──────────────────────────────
# Change this path to where YOUR csv is in Drive
DRIVE_CSV = "/content/drive/MyDrive/MBTI_Classifier/mbti_1.csv"

if os.path.exists(DRIVE_CSV):
    shutil.copy(DRIVE_CSV, "/content/mbti_1.csv")
    size = os.path.getsize("/content/mbti_1.csv") / (1024*1024)
    print(f"✅ mbti_1.csv copied from Drive! ({size:.1f} MB)")
elif os.path.exists("/content/mbti_1.csv"):
    print("✅ mbti_1.csv already in Colab!")
else:
    print("⚠️  mbti_1.csv not found in Drive path above!")
    print("   Uploading manually instead...")
    from google.colab import files
    files.upload()

# ── Download GloVe 50d (CPU optimized) ───────────────────────
if not os.path.exists("glove.6B.50d.txt"):
    print("\n⏳ Downloading GloVe 50d embeddings...")
    print("   This downloads once and takes ~1-2 minutes...")
    !wget -q --show-progress http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.50d.txt
    print("✅ GloVe 50d downloaded!")
else:
    print("✅ GloVe 50d already exists — skipping!")

# ── Verify both files ─────────────────────────────────────────
print("\nFile check:")
for f in ["/content/mbti_1.csv", "glove.6B.50d.txt"]:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024*1024)
        print(f"  ✅ {f.split('/')[-1]} — {size:.1f} MB")
    else:
        print(f"  ❌ Missing: {f}")

✅ mbti_1.csv copied from Drive! (59.9 MB)
✅ GloVe 50d already exists — skipping!

File check:
  ✅ mbti_1.csv — 59.9 MB
  ✅ glove.6B.50d.txt — 163.4 MB


In [56]:
class Config:
    # Paths
    DATA_PATH      = "/content/mbti_1.csv"
    GLOVE_PATH     = "glove.6B.50d.txt"
    MODEL_PATH     = "/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras"
    TOKENIZER_PATH = "/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl"

    # CPU optimized settings
    MAX_VOCAB      = 10_000
    MAX_LEN        = 64
    POSTS_PER_USER = 3
    EMBED_DIM      = 50

    # Lightweight model
    LSTM_UNITS     = 32
    DROPOUT        = 0.3
    REC_DROPOUT    = 0.1

    # Training
    BATCH_SIZE     = 256
    EPOCHS         = 5
    LR             = 1e-3
    TEST_SIZE      = 0.15
    VAL_SIZE       = 0.15
    SEED           = 42
    AXES           = ["IE", "NS", "TF", "JP"]

cfg = Config()
tf.random.set_seed(cfg.SEED)
np.random.seed(cfg.SEED)

print("✅ Config loaded!")
print(f"   MAX_LEN      = {cfg.MAX_LEN}")
print(f"   LSTM_UNITS   = {cfg.LSTM_UNITS}")
print(f"   BATCH_SIZE   = {cfg.BATCH_SIZE}")
print(f"   EMBED_DIM    = {cfg.EMBED_DIM}")
print(f"   EPOCHS       = {cfg.EPOCHS}")

✅ Config loaded!
   MAX_LEN      = 64
   LSTM_UNITS   = 32
   BATCH_SIZE   = 256
   EMBED_DIM    = 50
   EPOCHS       = 5


In [57]:
# ── Load data ─────────────────────────────────────────────────
def load_data(path):
    df = pd.read_csv(path)
    print(f"✅ Loaded {len(df):,} rows")
    axis_maps = [
        {"I":0,"E":1},
        {"N":0,"S":1},
        {"T":0,"F":1},
        {"J":0,"P":1}
    ]
    for i, (axis, mapping) in enumerate(zip(cfg.AXES, axis_maps)):
        df[axis] = df["type"].apply(lambda t: mapping[t[i]])
    print(f"   Top types:\n{df['type'].value_counts().head(5).to_string()}")
    return df

# ── Clean text ────────────────────────────────────────────────
def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(
        r"\b(infp|infj|intp|intj|enfp|enfj|entp|entj|"
        r"isfp|isfj|istp|istj|esfp|esfj|estp|estj)\b",
        "", text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-zA-Z\s.,!?']", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

def prepare_texts(df):
    texts = []
    for posts_raw in df["posts"]:
        posts   = posts_raw.split("|||")
        sampled = posts[:cfg.POSTS_PER_USER]
        texts.append(" ".join(clean_text(p) for p in sampled))
    return texts

# ── Run ───────────────────────────────────────────────────────
df     = load_data(cfg.DATA_PATH)
texts  = prepare_texts(df)
labels = df[cfg.AXES].values
print(f"\n✅ Text preparation done — {len(texts):,} samples")

# ── Split ─────────────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels,
    test_size=cfg.TEST_SIZE,
    random_state=cfg.SEED,
    stratify=labels[:,0]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=cfg.VAL_SIZE / (1 - cfg.TEST_SIZE),
    random_state=cfg.SEED,
    stratify=y_temp[:,0]
)
print(f"✅ Split → train={len(X_train):,} val={len(X_val):,} test={len(X_test):,}")

# ── Tokenize ──────────────────────────────────────────────────
tokenizer = Tokenizer(num_words=cfg.MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
print(f"✅ Vocabulary: {len(tokenizer.word_index):,} words")

def to_seq(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=cfg.MAX_LEN,
        padding="post",
        truncating="post"
    )

X_train_seq = to_seq(X_train)
X_val_seq   = to_seq(X_val)
X_test_seq  = to_seq(X_test)
print(f"✅ Sequences ready — shape: {X_train_seq.shape}")

# ── Load GloVe 50d ────────────────────────────────────────────
print("\n⏳ Loading GloVe 50d...")
glove = {}
with open(cfg.GLOVE_PATH, encoding="utf-8") as f:
    for line in tqdm(f, desc="GloVe"):
        v = line.split()
        glove[v[0]] = np.asarray(v[1:], dtype="float32")
print(f"✅ Loaded {len(glove):,} GloVe vectors")

# ── Embedding matrix ──────────────────────────────────────────
vocab_size   = min(len(tokenizer.word_index)+1, cfg.MAX_VOCAB+1)
embed_matrix = np.random.normal(
    scale=0.1, size=(vocab_size, cfg.EMBED_DIM)
).astype("float32")
embed_matrix[0] = 0

hits = 0
for word, idx in tokenizer.word_index.items():
    if idx < cfg.MAX_VOCAB and word in glove:
        embed_matrix[idx] = glove[word]
        hits += 1

coverage = hits / min(len(tokenizer.word_index), cfg.MAX_VOCAB) * 100
print(f"✅ Embedding matrix built — {coverage:.1f}% vocabulary covered")

# ── Label dicts ───────────────────────────────────────────────
def labels_to_dict(y):
    return {f"output_{ax}": y[:,i] for i, ax in enumerate(cfg.AXES)}

y_train_dict = labels_to_dict(y_train)
y_val_dict   = labels_to_dict(y_val)
y_test_dict  = labels_to_dict(y_test)

print("\n✅ All data preparation complete!")
print("   Ready to build and train the model.")

✅ Loaded 8,675 rows
   Top types:
type
INFP    1832
INFJ    1470
INTP    1304
INTJ    1091
ENTP     685

✅ Text preparation done — 8,675 samples
✅ Split → train=6,071 val=1,302 test=1,302
✅ Vocabulary: 23,058 words
✅ Sequences ready — shape: (6071, 64)

⏳ Loading GloVe 50d...


GloVe: 400000it [00:19, 20712.30it/s]


✅ Loaded 400,000 GloVe vectors
✅ Embedding matrix built — 93.1% vocabulary covered

✅ All data preparation complete!
   Ready to build and train the model.


In [ ]:
# ── Build lightweight CPU model ───────────────────────────────
def build_model(embedding_matrix, trainable_embed=False):
    vocab_size, embed_dim = embedding_matrix.shape
    inp = Input(shape=(cfg.MAX_LEN,), name="input_tokens")

    x = Embedding(
        vocab_size, embed_dim,
        weights=[embedding_matrix],
        input_length=cfg.MAX_LEN,
        trainable=trainable_embed,
        name="glove_embedding"
    )(inp)

    # Single BiLSTM layer (CPU friendly)
    x = Bidirectional(
        LSTM(cfg.LSTM_UNITS,
             return_sequences=True,
             dropout=cfg.DROPOUT,
             recurrent_dropout=cfg.REC_DROPOUT),
        name="bilstm_1"
    )(x)

    x = GlobalMaxPooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(cfg.DROPOUT)(x)

    # 4 output heads — one per MBTI axis
    outputs = [
        Dense(1, activation="sigmoid", name=f"output_{ax}")(x)
        for ax in cfg.AXES
    ]
    return Model(inputs=inp, outputs=outputs, name="MBTI_BiLSTM_CPU")

# ── Train function ────────────────────────────────────────────
def compile_and_train(model, X_tr, y_tr, X_v, y_v, lr=cfg.LR):
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss={f"output_{ax}": "binary_crossentropy" for ax in cfg.AXES},
        metrics={f"output_{ax}": ["accuracy"] for ax in cfg.AXES},
    )
    callbacks = [
        EarlyStopping(
            monitor="val_loss", patience=3,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=2, verbose=1
        ),
        ModelCheckpoint(
            cfg.MODEL_PATH, monitor="val_loss",
            save_best_only=True, verbose=0
        ),
    ]
    return model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=cfg.EPOCHS,
        batch_size=cfg.BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

# ── Run training ──────────────────────────────────────────────
print("🏗️  Building model...")
model = build_model(embed_matrix, trainable_embed=False)
model.summary()

print("\n🚀 Training started!")
print("   Estimated time on CPU: ~20-30 minutes")
print("   EarlyStopping will stop early if model converges")
print("─" * 55)

history = compile_and_train(
    model,
    X_train_seq, y_train_dict,
    X_val_seq,   y_val_dict
)

print("\n✅ Training complete!")
print("   Run CELL 9 immediately to save the model!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


🏗️  Building model...


Model: "MBTI_BiLSTM_CPU"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tokens        │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glove_embedding     │ (None, 64, 50)    │    500,050 │ input_tokens[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 64, 64)    │     21,248 │ glove_embedding[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ bilstm_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,160 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_IE (Dense)   │ (None, 1)         │         65 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_NS (Dense)   │ (None, 1)         │         65 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_TF (Dense)   │ (None, 1)         │         65 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_JP (Dense)   │ (None, 1)         │         65 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 525,718 (2.01 MB)

 Trainable params: 25,668 (100.27 KB)

 Non-trainable params: 500,050 (1.91 MB)


🚀 Training started!
   Estimated time on CPU: ~20-30 minutes
   EarlyStopping will stop early if model converges
───────────────────────────────────────────────────────
Epoch 1/5
17/24 ━━━━━━━━━━━━━━━━━━━━ 1s 255ms/step - loss: 2.6589 - output_IE_accuracy: 0.5138 - output_IE_loss: 0.6909 - output_JP_accuracy: 0.5407 - output_JP_loss: 0.6873 - output_NS_accuracy: 0.7635 - output_NS_loss: 0.5820 - output_TF_accuracy: 0.5136 - output_TF_loss: 0.6987

In [ ]:
import pickle

print("💾 Saving to Google Drive...")

# Save model
model.save(cfg.MODEL_PATH)
print(f"✅ Model saved!")

# Save tokenizer
with open(cfg.TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)
print(f"✅ Tokenizer saved!")

# Verify saved files
for path in [cfg.MODEL_PATH, cfg.TOKENIZER_PATH]:
    size = os.path.getsize(path) / (1024*1024)
    print(f"   📁 {path.split('/')[-1]} — {size:.1f} MB")

print("\n🎉 Model saved permanently to Drive!")
print("   Next time open Colab → skip to CELL 10!")

In [ ]:
import pickle
import tensorflow as tf
from google.colab import drive
import os

# Mount drive if not already
try:
    drive.mount('/content/drive')
except:
    pass

MODEL_PATH     = "/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras"
TOKENIZER_PATH = "/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl"

print("⏳ Loading model from Drive...")
model = tf.keras.models.load_model(MODEL_PATH)
print("✅ Model loaded!")

print("⏳ Loading tokenizer...")
with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)
print("✅ Tokenizer loaded!")

print("\n🎉 Ready! Jump to CELL 12 to predict.")
print(f"   Model input shape: {model.input_shape}")

In [ ]:
print("📊 Evaluating on test set...")

preds_raw = model.predict(X_test_seq, batch_size=256, verbose=0)

fig, axes_plots = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("MBTI BiLSTM — Confusion Matrices", fontsize=14)

AXES       = ["IE", "NS", "TF", "JP"]
macro_f1s  = []

for i, ax_name in enumerate(AXES):
    y_true  = y_test_dict[f"output_{ax_name}"]
    y_pred  = (preds_raw[i].squeeze() >= 0.5).astype(int)
    letters = [ax_name[0], ax_name[1]]

    print(f"\n{'─'*45}")
    print(f"Axis {ax_name}  ({ax_name[0]}=0  {ax_name[1]}=1)")
    print(classification_report(y_true, y_pred, target_names=letters))

    mf1 = f1_score(y_true, y_pred, average="macro")
    macro_f1s.append(mf1)

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=letters, yticklabels=letters,
                ax=axes_plots[i])
    axes_plots[i].set_title(f"Axis {ax_name} | F1={mf1:.3f}")

print(f"\n{'═'*45}")
print(f"✅ Mean macro-F1 across 4 axes: {np.mean(macro_f1s):.4f}")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# Save to Drive
import shutil
shutil.copy(
    "confusion_matrices.png",
    "/content/drive/MyDrive/MBTI_Classifier/assets/confusion_matrices.png"
)
print("✅ Saved confusion_matrices.png to Drive!")

In [ ]:
import re
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 64
AXES    = ["IE", "NS", "TF", "JP"]

MBTI_NAMES = {
    "INTJ":"The Architect",  "INTP":"The Thinker",
    "ENTJ":"The Commander",  "ENTP":"The Debater",
    "INFJ":"The Advocate",   "INFP":"The Mediator",
    "ENFJ":"The Protagonist","ENFP":"The Campaigner",
    "ISTJ":"The Logistician","ISFJ":"The Defender",
    "ESTJ":"The Executive",  "ESFJ":"The Consul",
    "ISTP":"The Virtuoso",   "ISFP":"The Adventurer",
    "ESTP":"The Entrepreneur","ESFP":"The Entertainer",
}

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(
        r"\b(infp|infj|intp|intj|enfp|enfj|entp|entj|"
        r"isfp|isfj|istp|istj|esfp|esfj|estp|estj)\b",
        "", text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-zA-Z\s.,!?']", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

def predict_mbti(text):
    cleaned  = clean_text(text)
    seq      = tokenizer.texts_to_sequences([cleaned])
    padded   = pad_sequences(seq, maxlen=MAX_LEN,
                             padding="post", truncating="post")
    probs    = model.predict(padded, verbose=0)
    axis_map = [("I","E"),("N","S"),("T","F"),("J","P")]
    letters  = [
        m[1] if probs[i][0][0] >= 0.5 else m[0]
        for i, m in enumerate(axis_map)
    ]
    mbti = "".join(letters)
    name = MBTI_NAMES.get(mbti, "Unknown")

    print(f"\n{'═'*55}")
    print(f"  📝 Input  : {text[:70]}{'...' if len(text)>70 else ''}")
    print(f"  🧠 Type   : {mbti} — {name}")
    print(f"{'─'*55}")
    print(f"  IE={probs[0][0][0]:.2f} "
          f"NS={probs[1][0][0]:.2f} "
          f"TF={probs[2][0][0]:.2f} "
          f"JP={probs[3][0][0]:.2f}")
    print(f"  (score >0.5 = E/S/F/P,  score <0.5 = I/N/T/J)")
    print(f"{'═'*55}")
    return mbti

# ── Test samples ──────────────────────────────────────────────
predict_mbti("I love spending time alone reading and thinking about abstract ideas.")
predict_mbti("I enjoy parties, meeting new people and talking about practical things.")
predict_mbti("Logic and data drive all my decisions, emotions are not relevant.")
predict_mbti("I care deeply about others and consider feelings in every decision.")

# # ── Try your own text ─────────────────────────────────────────
# print("\n" + "─"*55)
# print("✏️  Enter your own text below:")
# my_text = input("Your text: ")
# if my_text.strip():
#     predict_mbti(my_text)

In [ ]:
# # See Streamlit errors
# import subprocess
# result = subprocess.run(
#     ["streamlit", "run", "/content/mbti-streamlit/app.py",
#      "--server.port", "8501",
#      "--server.headless", "true"],
#     capture_output=True, text=True, timeout=15
# )
# print("STDOUT:", result.stdout)
# print("STDERR:", result.stderr)

In [ ]:
import os

# Check if the folder exists at all
if os.path.exists("/content/mbti-streamlit"):
    print("✅ mbti-streamlit folder exists")
    print("\nContents:")
    for root, dirs, files in os.walk("/content/mbti-streamlit"):
        level = root.replace("/content/mbti-streamlit", "").count(os.sep)
        indent = "  " * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        for file in files:
            print(f"{indent}  📄 {file}")
else:
    print("❌ mbti-streamlit folder does NOT exist!")
    print("   Need to create it from scratch")

In [ ]:
import os

folders = [
    "/content/mbti-streamlit",
    "/content/mbti-streamlit/pages",
    "/content/mbti-streamlit/utils",
    "/content/mbti-streamlit/model",
    "/content/mbti-streamlit/assets",
]
for f in folders:
    os.makedirs(f, exist_ok=True)

# Create empty __init__ files
open("/content/mbti-streamlit/pages/__init__.py", "w").close()
open("/content/mbti-streamlit/utils/__init__.py", "w").close()

print("✅ Folder structure created!")
for f in folders:
    print(f"   📁 {f}")

In [ ]:
mbti_data_code = '''
MBTI_TYPES = {
    "INTJ": {"name":"The Architect","emoji":"🏛️","color":"#6e9ec8",
             "tagline":"Imaginative and strategic thinkers with a plan for everything.",
             "description":"INTJs are analytical problem-solvers driven by original ideas.",
             "strengths":["Strategic","Independent","Decisive","Confident"],
             "weaknesses":["Arrogant","Dismissive","Overly critical"],
             "famous":["Elon Musk","Stephen Hawking","Nikola Tesla"],
             "careers":["Software Architect","Scientist","Strategist"]},
    "INTP": {"name":"The Thinker","emoji":"💡","color":"#6e9ec8",
             "tagline":"Innovative inventors with an unquenchable thirst for knowledge.",
             "description":"INTPs are analytical and objective, valuing logic above all.",
             "strengths":["Logical","Original","Objective","Curious"],
             "weaknesses":["Insensitive","Absent-minded","Indecisive"],
             "famous":["Albert Einstein","Bill Gates","Isaac Newton"],
             "careers":["Researcher","Developer","Mathematician"]},
    "ENTJ": {"name":"The Commander","emoji":"⚔️","color":"#c8a96e",
             "tagline":"Bold, imaginative leaders who always find a way.",
             "description":"ENTJs are natural-born leaders who embody rationality.",
             "strengths":["Efficient","Energetic","Strategic","Confident"],
             "weaknesses":["Stubborn","Dominant","Intolerant"],
             "famous":["Steve Jobs","Napoleon","Margaret Thatcher"],
             "careers":["CEO","Entrepreneur","Lawyer"]},
    "ENTP": {"name":"The Debater","emoji":"🎯","color":"#c8a96e",
             "tagline":"Smart and curious thinkers who love a challenge.",
             "description":"ENTPs are quick-thinking individuals who love to debate.",
             "strengths":["Knowledgeable","Original","Quick thinker"],
             "weaknesses":["Argumentative","Insensitive","Difficulty focusing"],
             "famous":["Mark Twain","Leonardo da Vinci","Socrates"],
             "careers":["Entrepreneur","Lawyer","Consultant"]},
    "INFJ": {"name":"The Advocate","emoji":"🌿","color":"#a8c88e",
             "tagline":"Quiet visionaries who inspire others with creative ideas.",
             "description":"INFJs are idealistic and principled with strong integrity.",
             "strengths":["Creative","Insightful","Principled","Passionate"],
             "weaknesses":["Sensitive","Perfectionist","Burnout-prone"],
             "famous":["Nelson Mandela","Mother Teresa","Plato"],
             "careers":["Counselor","Writer","Psychologist"]},
    "INFP": {"name":"The Mediator","emoji":"🦋","color":"#a8c88e",
             "tagline":"Poetic, kind and altruistic people, always eager to help.",
             "description":"INFPs are imaginative idealists guided by core values.",
             "strengths":["Empathetic","Generous","Creative","Passionate"],
             "weaknesses":["Unrealistic","Overly emotional","Impractical"],
             "famous":["J.R.R. Tolkien","Shakespeare","Audrey Hepburn"],
             "careers":["Writer","Counselor","Artist"]},
    "ENFJ": {"name":"The Protagonist","emoji":"🌟","color":"#a8c88e",
             "tagline":"Charismatic and inspiring leaders.",
             "description":"ENFJs are people-focused, warm, and highly organized.",
             "strengths":["Charismatic","Reliable","Altruistic","Tolerant"],
             "weaknesses":["Too selfless","Too sensitive","Overly idealistic"],
             "famous":["Barack Obama","Oprah Winfrey","Jennifer Lawrence"],
             "careers":["Teacher","Coach","HR Manager"]},
    "ENFP": {"name":"The Campaigner","emoji":"🎨","color":"#c8a96e",
             "tagline":"Enthusiastic creative souls who see life as full of possibilities.",
             "description":"ENFPs are creative, energetic free spirits.",
             "strengths":["Curious","Energetic","Creative","Sociable"],
             "weaknesses":["Overthinks","Highly emotional","Difficulty focusing"],
             "famous":["Robin Williams","Ellen DeGeneres","Walt Disney"],
             "careers":["Journalist","Actor","Entrepreneur"]},
    "ISTJ": {"name":"The Logistician","emoji":"📋","color":"#9ec8c8",
             "tagline":"Practical and fact-minded, reliable above all.",
             "description":"ISTJs are responsible organizers driven to create order.",
             "strengths":["Honest","Responsible","Dedicated","Calm"],
             "weaknesses":["Stubborn","Insensitive","Judgmental"],
             "famous":["Queen Elizabeth II","George Washington","Warren Buffett"],
             "careers":["Accountant","Military","Judge"]},
    "ISFJ": {"name":"The Defender","emoji":"🛡️","color":"#9ec8c8",
             "tagline":"Dedicated and warm protectors.",
             "description":"ISFJs are industrious caretakers loyal to traditions.",
             "strengths":["Supportive","Reliable","Patient","Loyal"],
             "weaknesses":["Humble","Takes things personally","Reluctant to change"],
             "famous":["Mother Teresa","Beyoncé","Kate Middleton"],
             "careers":["Nurse","Teacher","Social Worker"]},
    "ESTJ": {"name":"The Executive","emoji":"👔","color":"#9ec8c8",
             "tagline":"Excellent administrators, unsurpassed at managing.",
             "description":"ESTJs are hardworking traditionalists eager to take charge.",
             "strengths":["Dedicated","Direct","Honest","Loyal"],
             "weaknesses":["Inflexible","Judgmental","Bossy"],
             "famous":["Judge Judy","Frank Sinatra","Michelle Obama"],
             "careers":["Manager","Judge","Financial Officer"]},
    "ESFJ": {"name":"The Consul","emoji":"🤝","color":"#9ec8c8",
             "tagline":"Caring, social people always eager to help.",
             "description":"ESFJs value harmony and are warm and sociable.",
             "strengths":["Loyal","Sensitive","Warm","Practical"],
             "weaknesses":["Inflexible","Vulnerable to criticism"],
             "famous":["Taylor Swift","Bill Clinton","Jennifer Garner"],
             "careers":["Teacher","Healthcare","HR"]},
    "ISTP": {"name":"The Virtuoso","emoji":"🔧","color":"#c89e6e",
             "tagline":"Bold and practical experimenters.",
             "description":"ISTPs are observant artisans with mechanical skill.",
             "strengths":["Practical","Creative","Spontaneous","Rational"],
             "weaknesses":["Stubborn","Insensitive","Easily bored"],
             "famous":["Clint Eastwood","Michael Jordan","Kobe Bryant"],
             "careers":["Engineer","Mechanic","Pilot"]},
    "ISFP": {"name":"The Adventurer","emoji":"🌸","color":"#c89e6e",
             "tagline":"Flexible and charming artists.",
             "description":"ISFPs are gentle caregivers who live in the present.",
             "strengths":["Charming","Imaginative","Passionate","Curious"],
             "weaknesses":["Unpredictable","Easily stressed"],
             "famous":["Michael Jackson","David Bowie","Lana Del Rey"],
             "careers":["Artist","Musician","Designer"]},
    "ESTP": {"name":"The Entrepreneur","emoji":"⚡","color":"#c89e6e",
             "tagline":"Energetic thrill-seekers at their best under pressure.",
             "description":"ESTPs are bold thrill-seekers who love real-world action.",
             "strengths":["Bold","Rational","Practical","Sociable"],
             "weaknesses":["Impatient","Risk-prone","Insensitive"],
             "famous":["Ernest Hemingway","Madonna","Jack Nicholson"],
             "careers":["Entrepreneur","Stockbroker","Detective"]},
    "ESFP": {"name":"The Entertainer","emoji":"🎭","color":"#c89e6e",
             "tagline":"Spontaneous entertainers — life is never boring around them.",
             "description":"ESFPs are vivacious entertainers who charm everyone.",
             "strengths":["Bold","Original","Practical","Showmanship"],
             "weaknesses":["Sensitive","Easily bored","Poor long-term planning"],
             "famous":["Adele","Marilyn Monroe","Elvis Presley"],
             "careers":["Performer","Event Planner","Sales"]},
}

AXES_INFO = {
    "IE": {"label":"Energy",      "left":"Introvert","left_short":"I","right":"Extravert","right_short":"E"},
    "NS": {"label":"Information", "left":"Intuitive","left_short":"N","right":"Sensing",  "right_short":"S"},
    "TF": {"label":"Decisions",   "left":"Thinking", "left_short":"T","right":"Feeling",  "right_short":"F"},
    "JP": {"label":"Lifestyle",   "left":"Judging",  "left_short":"J","right":"Perceiving","right_short":"P"},
}
'''

with open("/content/mbti-streamlit/utils/mbti_data.py", "w") as f:
    f.write(mbti_data_code)

print("✅ utils/mbti_data.py created!")

In [ ]:
predictor_code = '''
import re
import numpy as np
import pickle
import os
import streamlit as st

MAX_LEN   = 64
AXES      = ["IE", "NS", "TF", "JP"]
THRESHOLD = 0.5

def clean_text(text):
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(
        r"\\b(infp|infj|intp|intj|enfp|enfj|entp|entj|"
        r"isfp|isfj|istp|istj|esfp|esfj|estp|estj)\\b",
        "", text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-zA-Z\\s.,!?\\']", " ", text)
    return re.sub(r"\\s+", " ", text).strip().lower()

@st.cache_resource(show_spinner="Loading model...")
def load_model_and_tokenizer(model_path, tokenizer_path):
    import tensorflow as tf
    from tensorflow.keras.models import load_model
    if not os.path.exists(model_path) or not os.path.exists(tokenizer_path):
        return None, None
    model = load_model(model_path)
    with open(tokenizer_path, "rb") as f:
        tokenizer = pickle.load(f)
    return model, tokenizer

def predict(text, model, tokenizer):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    cleaned = clean_text(text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    raw     = model.predict(padded, verbose=0)
    probs, letters = {}, {}
    axis_map = [("I","E"),("N","S"),("T","F"),("J","P")]
    for i, ax in enumerate(AXES):
        p = float(raw[i][0][0])
        probs[ax]   = p
        letters[ax] = axis_map[i][1] if p >= THRESHOLD else axis_map[i][0]
    mbti_type  = "".join(letters[ax] for ax in AXES)
    confidence = float(np.mean([
        probs["IE"] if letters["IE"]=="E" else 1-probs["IE"],
        probs["NS"] if letters["NS"]=="S" else 1-probs["NS"],
        probs["TF"] if letters["TF"]=="F" else 1-probs["TF"],
        probs["JP"] if letters["JP"]=="P" else 1-probs["JP"],
    ]))
    return {"type":mbti_type,"probs":probs,"letters":letters,"confidence":confidence}
'''

with open("/content/mbti-streamlit/utils/predictor.py", "w") as f:
    f.write(predictor_code)

print("✅ utils/predictor.py created!")

In [ ]:
app_code = '''
import streamlit as st

st.set_page_config(
    page_title="MBTI Personality Classifier",
    page_icon="",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
@import url(\'https://fonts.googleapis.com/css2?family=DM+Serif+Display&family=DM+Sans:wght@300;400;600&display=swap\');
html, body, [class*="css"] { font-family: \'DM Sans\', sans-serif; background:#0d0f14 !important; color:#e8e4dc !important; }
.stApp { background:#0d0f14; }
h1,h2,h3 { font-family: \'DM Serif Display\', serif; color:#e8e4dc !important; }
[data-testid="stSidebar"] { background:#151820 !important; border-right:1px solid #1e2330; }
.stButton > button { background:#c8a96e !important; color:#0d0f14 !important; border:none !important; font-weight:600 !important; border-radius:4px !important; }
.stTextArea textarea { background:#151820 !important; border:1px solid #1e2330 !important; color:#e8e4dc !important; }
.stSelectbox div { background:#151820 !important; color:#e8e4dc !important; }
.mbti-card { background:#151820; border:1px solid #1e2330; border-radius:8px; padding:1.5rem; margin:0.5rem 0; }
.metric-box { background:#151820; border:1px solid #1e2330; border-radius:8px; padding:1rem; text-align:center; }
.metric-val { font-family:\'DM Serif Display\',serif; font-size:2.5rem; color:#c8a96e; }
.metric-lbl { font-size:0.75rem; color:#6b7280; text-transform:uppercase; letter-spacing:0.1em; }
#MainMenu, footer { visibility:hidden; }
</style>
""", unsafe_allow_html=True)

with st.sidebar:
    st.markdown("""
    <div style=\'padding:1rem 0\'>
        <div style=\'font-family:DM Serif Display,serif;font-size:1.6rem;color:#c8a96e\'> MBTI Classifier</div>
        <div style=\'color:#6b7280;font-size:0.8rem;margin-top:0.3rem\'>NLP with Deep Learning · BiLSTM</div>
    </div>
    """, unsafe_allow_html=True)
    st.markdown("---")
    page = st.radio("Navigate", [
        " Home",
        " Personality Predictor",
        " Analytics Dashboard",
        " Batch Testing",
        " MBTI Explorer"
    ], label_visibility="collapsed")
    st.markdown("---")
    st.markdown("""
    <div style=\'color:#6b7280;font-size:0.75rem;line-height:1.6\'>
        <b style=\'color:#c8a96e\'>Model:</b> Bidirectional LSTM<br>
        <b style=\'color:#c8a96e\'>Embeddings:</b> GloVe 50d<br>
        <b style=\'color:#c8a96e\'>Task:</b> 4-axis classification<br>
        <b style=\'color:#c8a96e\'>Dataset:</b> MBTI Kaggle (8,675)
    </div>
    """, unsafe_allow_html=True)

if page == " Home":
    from pages.home import render; render()
elif page == " Personality Predictor":
    from pages.predictor import render; render()
elif page == " Analytics Dashboard":
    from pages.analytics import render; render()
elif page == " Batch Testing":
    from pages.batch import render; render()
elif page == " MBTI Explorer":
    from pages.explorer import render; render()
'''

with open("/content/mbti-streamlit/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created!")

In [ ]:
# ── pages/home.py ─────────────────────────────────────────────
home_code = '''
import streamlit as st
def render():
    st.markdown("<div style=\'font-family:DM Serif Display,serif;font-size:3rem;color:#c8a96e\'>MBTI Personality Classifier</div>", unsafe_allow_html=True)
    st.markdown("<div style=\'color:#6b7280;margin-top:0.5rem\'>Bidirectional LSTM model trained on 8,675 posts to predict personality across 4 axes.</div>", unsafe_allow_html=True)
    st.markdown("---")
    c1,c2,c3,c4 = st.columns(4)
    for col, val, lbl in zip([c1,c2,c3,c4],
        ["8,675","4","3.4M","16"],
        ["Training Samples","MBTI Axes","Parameters","Personality Types"]):
        col.markdown(f"<div class=\'metric-box\'><div class=\'metric-val\'>{val}</div><div class=\'metric-lbl\'>{lbl}</div></div>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("### The 4 MBTI Axes")
    c1,c2,c3,c4 = st.columns(4)
    for col, letters, cat, color in zip([c1,c2,c3,c4],
        ["I / E","N / S","T / F","J / P"],
        ["Energy","Information","Decisions","Lifestyle"],
        ["#6e9ec8","#c8a96e","#a8c88e","#c89e6e"]):
        col.markdown(f"<div class=\'mbti-card\'><div style=\'font-family:DM Serif Display,serif;font-size:1.8rem;color:{color}\'>{letters}</div><div style=\'font-size:0.7rem;color:#6b7280;text-transform:uppercase\'>{cat}</div></div>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)
    st.info(" Use the sidebar to navigate to the Personality Predictor!")
'''

# ── pages/predictor.py ────────────────────────────────────────
predictor_page_code = '''
import streamlit as st
import plotly.graph_objects as go
import numpy as np, hashlib
from utils.mbti_data import MBTI_TYPES, AXES_INFO

MODEL_PATH     = "model/mbti_bilstm_model.keras"
TOKENIZER_PATH = "model/tokenizer.pkl"

DEMOS = {
    "Introvert Thinker": "I prefer spending evenings alone reading technical papers. Logic and data drive every decision I make. I find small talk exhausting.",
    "Extravert Feeler":  "I absolutely love meeting new people and hearing their stories! Every interaction fills me with energy. I care deeply about how others feel.",
    "Structured Planner":"I always have a detailed plan before starting any project. I believe in following rules and meeting every deadline.",
    "Spontaneous Explorer":"I love living in the moment and exploring new places without any plan. I adapt quickly and find unexpected situations exciting.",
}

def demo_predict(text):
    h = int(hashlib.md5(text.encode()).hexdigest(), 16)
    probs = {"IE":(h%100)/100,"NS":((h>>8)%100)/100,"TF":((h>>16)%100)/100,"JP":((h>>24)%100)/100}
    letters = {"IE":"E" if probs["IE"]>=0.5 else "I","NS":"S" if probs["NS"]>=0.5 else "N",
               "TF":"F" if probs["TF"]>=0.5 else "T","JP":"P" if probs["JP"]>=0.5 else "J"}
    axes = ["IE","NS","TF","JP"]
    confidence = float(np.mean([probs[ax] if letters[ax] in ["E","S","F","P"] else 1-probs[ax] for ax in axes]))
    return {"type":"".join(letters[ax] for ax in axes),"probs":probs,"letters":letters,"confidence":confidence}

def render():
    st.markdown("<div style=\'font-family:DM Serif Display,serif;font-size:2.2rem;color:#c8a96e\'>🔮 Personality Predictor</div>", unsafe_allow_html=True)
    st.markdown("<div style=\'color:#6b7280\'>Enter any text and the BiLSTM model will predict your MBTI type.</div>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    try:
        from utils.predictor import load_model_and_tokenizer, predict
        model, tokenizer = load_model_and_tokenizer(MODEL_PATH, TOKENIZER_PATH)
        model_loaded = model is not None
    except:
        model_loaded = False

    if not model_loaded:


    demo_choice = st.selectbox("Try a sample:", ["(write your own...)"] + list(DEMOS.keys()))
    text_input  = st.text_area("Your text:", value=DEMOS.get(demo_choice,""), height=150,
                                placeholder="Type or paste your text here...")
    run = st.button(" Predict My Personality")

    if run and text_input.strip():
        with st.spinner("Analysing..."):
            result = (predict(text_input, model, tokenizer) if model_loaded else demo_predict(text_input))

        mbti  = result["type"]
        info  = MBTI_TYPES.get(mbti, {})
        color = info.get("color","#c8a96e")

        st.markdown("---")
        c1, c2 = st.columns([1,2])

        with c1:
            st.markdown(f"""
            <div class=\'mbti-card\' style=\'text-align:center;padding:2rem\'>
                <div style=\'color:#6b7280;font-size:0.8rem;text-transform:uppercase;letter-spacing:0.15em\'>You are</div>
                <div style=\'font-family:DM Serif Display,serif;font-size:5rem;color:{color};line-height:1\'>{mbti}</div>
                <div style=\'font-size:1.2rem;margin-top:0.5rem\'>{info.get("emoji","")} {info.get("name","")}</div>
                <div style=\'color:#6b7280;font-size:0.85rem;font-style:italic;margin-top:0.3rem\'>"{info.get("tagline","")}"</div>
                <div style=\'margin-top:1rem;color:#c8a96e;font-weight:600\'>{result["confidence"]*100:.0f}% confidence</div>
            </div>
            """, unsafe_allow_html=True)

        with c2:
            st.markdown("#### Axis Confidence")
            axes_order = ["IE","NS","TF","JP"]
            axis_map   = [("I","E"),("N","S"),("T","F"),("J","P")]
            for ax,(l0,l1) in zip(axes_order,axis_map):
                p      = result["probs"][ax]
                chosen = result["letters"][ax]
                pct    = p if chosen==l1 else 1-p
                c_l,c_m,c_r = st.columns([1,4,1])
                c_l.markdown(f"<div style=\'text-align:right;font-weight:700;padding-top:4px;color:{"#c8a96e" if chosen==l0 else "#6b7280"}\'>{l0}</div>", unsafe_allow_html=True)
                c_m.markdown(f"<div style=\'padding-top:6px\'><div style=\'background:#1e2330;border-radius:4px;height:10px\'><div style=\'background:{color};border-radius:4px;height:10px;width:{pct*100:.0f}%\'></div></div><div style=\'font-size:0.72rem;color:#6b7280;margin-top:2px\'>{ax} · {pct*100:.0f}% {chosen}</div></div>", unsafe_allow_html=True)
                c_r.markdown(f"<div style=\'font-weight:700;padding-top:4px;color:{"#c8a96e" if chosen==l1 else "#6b7280"}\'>{l1}</div>", unsafe_allow_html=True)

        if info:
            st.markdown("#### About Your Type")
            d1,d2,d3 = st.columns(3)
            with d1:
                st.markdown(f"<div class=\'mbti-card\'><b>Description</b><br><span style=\'color:#9ca3af;font-size:0.9rem\'>{info.get('description','')}</span></div>", unsafe_allow_html=True)
            with d2:
                s = "".join(f"<div style=\'color:#4ade80;font-size:0.85rem\'>✓ {x}</div>" for x in info.get("strengths",[]))
                st.markdown(f"<div class=\'mbti-card\'><b>Strengths</b><br>{s}</div>", unsafe_allow_html=True)
            with d3:
                f2 = "".join(f"<div style=\'padding:3px 0;border-bottom:1px solid #1e2330;font-size:0.85rem\'>⭐ {x}</div>" for x in info.get("famous",[]))
                st.markdown(f"<div class=\'mbti-card\'><b>Famous {mbti}s</b><br>{f2}</div>", unsafe_allow_html=True)
    elif run:
        st.warning("Please enter some text first!")
'''

# ── pages/analytics.py ───────────────────────────────────────
analytics_code = '''
import streamlit as st
import plotly.graph_objects as go
import numpy as np, hashlib
from utils.mbti_data import MBTI_TYPES

MODEL_PATH     = "model/mbti_bilstm_model.keras"
TOKENIZER_PATH = "model/tokenizer.pkl"

def demo_predict(text):
    h = int(hashlib.md5(text.encode()).hexdigest(), 16)
    probs = {"IE":(h%100)/100,"NS":((h>>8)%100)/100,"TF":((h>>16)%100)/100,"JP":((h>>24)%100)/100}
    letters = {"IE":"E" if probs["IE"]>=0.5 else "I","NS":"S" if probs["NS"]>=0.5 else "N",
               "TF":"F" if probs["TF"]>=0.5 else "T","JP":"P" if probs["JP"]>=0.5 else "J"}
    axes = ["IE","NS","TF","JP"]
    return {"type":"".join(letters[ax] for ax in axes),"probs":probs,"letters":letters,
            "confidence":float(np.mean([probs[ax] if letters[ax] in ["E","S","F","P"] else 1-probs[ax] for ax in axes]))}

def render():
    st.markdown("<div style=\'font-family:DM Serif Display,serif;font-size:2.2rem;color:#c8a96e\'>📊 Analytics Dashboard</div>", unsafe_allow_html=True)
    text_input = st.text_area("Enter text for analysis:", height=120, placeholder="Paste your text here...")
    if st.button(" Analyse"):
        if not text_input.strip():
            st.warning("Please enter some text!"); return
        try:
            from utils.predictor import load_model_and_tokenizer, predict
            model, tokenizer = load_model_and_tokenizer(MODEL_PATH, TOKENIZER_PATH)
            result = predict(text_input, model, tokenizer) if model else demo_predict(text_input)
        except:
            result = demo_predict(text_input)

        mbti  = result["type"]
        info  = MBTI_TYPES.get(mbti,{})
        color = info.get("color","#c8a96e")
        st.markdown("---")

        c1,c2,c3,c4 = st.columns(4)
        for col,ax,(l0,l1) in zip([c1,c2,c3,c4],["IE","NS","TF","JP"],[("I","E"),("N","S"),("T","F"),("J","P")]):
            chosen = result["letters"][ax]
            pct    = result["probs"][ax] if chosen==l1 else 1-result["probs"][ax]
            col.markdown(f"<div class=\'metric-box\'><div class=\'metric-val\' style=\'color:{color}\'>{chosen}</div><div class=\'metric-lbl\'>{ax} Axis</div><div style=\'font-size:0.85rem\'>{pct*100:.0f}%</div></div>", unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        cl, cr = st.columns(2)
        with cl:
            st.markdown("##### Radar Chart")
            cats = ["Extravert","Sensing","Feeling","Perceiving","Extravert"]
            vals = [result["probs"]["IE"]*100, result["probs"]["NS"]*100,
                    result["probs"]["TF"]*100, result["probs"]["JP"]*100,
                    result["probs"]["IE"]*100]
            fig = go.Figure(go.Scatterpolar(r=vals,theta=cats,fill="toself",
                            fillcolor="rgba(200,169,110,0.15)",line=dict(color=color,width=2)))
            fig.update_layout(polar=dict(bgcolor="#151820",radialaxis=dict(range=[0,100],gridcolor="#1e2330"),
                              angularaxis=dict(gridcolor="#1e2330")),
                              paper_bgcolor="#151820",font_color="#e8e4dc",height=300,margin=dict(t=20,b=20,l=40,r=40))
            st.plotly_chart(fig, use_container_width=True)
        with cr:
            st.markdown("##### Confidence Bars")
            vals2 = [result["probs"][ax]*100 for ax in ["IE","NS","TF","JP"]]
            fig2  = go.Figure(go.Bar(x=vals2,y=["I↔E","N↔S","T↔F","J↔P"],orientation="h",
                              marker_color=color,text=[f"{v:.0f}%" for v in vals2],textposition="outside"))
            fig2.add_vline(x=50,line_dash="dash",line_color="#6b7280")
            fig2.update_layout(paper_bgcolor="#151820",plot_bgcolor="#151820",font_color="#e8e4dc",
                               xaxis=dict(range=[0,115]),height=300,margin=dict(t=20,b=20))
            st.plotly_chart(fig2, use_container_width=True)

        st.markdown("##### All 16 Types")
        all_types = list(MBTI_TYPES.keys())
        for row in [all_types[i:i+4] for i in range(0,16,4)]:
            cols = st.columns(4)
            for col,t in zip(cols,row):
                is_match = t==mbti
                bg = color if is_match else "#1e2330"
                tc = "#0d0f14" if is_match else "#6b7280"
                col.markdown(f"<div style=\'background:{bg};border-radius:6px;padding:0.6rem;text-align:center;margin:2px\'><div style=\'font-family:DM Serif Display,serif;font-size:1.1rem;color:{tc}\'>{t}</div><div style=\'font-size:0.65rem;color:{tc}\'>{MBTI_TYPES[t]["name"]}</div></div>", unsafe_allow_html=True)
'''

# ── pages/batch.py ────────────────────────────────────────────
batch_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import hashlib
import plotly.express as px
from utils.mbti_data import MBTI_TYPES

MODEL_PATH     = "model/mbti_bilstm_model.keras"
TOKENIZER_PATH = "model/tokenizer.pkl"

def demo_predict(text):
    h = int(hashlib.md5(str(text).encode()).hexdigest(), 16)
    probs = {"IE":(h%100)/100,"NS":((h>>8)%100)/100,"TF":((h>>16)%100)/100,"JP":((h>>24)%100)/100}
    letters = {"IE":"E" if probs["IE"]>=0.5 else "I","NS":"S" if probs["NS"]>=0.5 else "N",
               "TF":"F" if probs["TF"]>=0.5 else "T","JP":"P" if probs["JP"]>=0.5 else "J"}
    axes = ["IE","NS","TF","JP"]
    return {"type":"".join(letters[ax] for ax in axes),"probs":probs,"letters":letters,"confidence":0.7}

def render():
    st.markdown("<div style=\'font-family:DM Serif Display,serif;font-size:2.2rem;color:#c8a96e\'> Batch Testing</div>", unsafe_allow_html=True)
    st.markdown("<div style=\'color:#6b7280\'>Upload a CSV with a text column to predict MBTI for all rows.</div>", unsafe_allow_html=True)

    sample_df = pd.DataFrame({"id":[1,2,3],"text":["I love spending time alone thinking about abstract ideas.","I enjoy parties and meeting new people every day.","Logic drives all my decisions, facts matter most."]})
    st.download_button(" Download Sample CSV", sample_df.to_csv(index=False).encode(), "sample.csv", "text/csv")

    uploaded = st.file_uploader("Upload your CSV", type=["csv"])
    if uploaded:
        df = pd.read_csv(uploaded)
        st.markdown(f" Loaded **{len(df):,} rows**")
        st.dataframe(df.head(3), use_container_width=True)

        text_col = next((c for c in ["text","posts","content","message"] if c in df.columns), df.columns[0])

        if st.button(f" Predict all {len(df):,} rows"):
            try:
                from utils.predictor import load_model_and_tokenizer, predict
                model, tokenizer = load_model_and_tokenizer(MODEL_PATH, TOKENIZER_PATH)
                use_model = model is not None
            except:
                use_model = False

            progress = st.progress(0, text="Predicting...")
            results  = []
            for i, row in enumerate(df[text_col].fillna("").tolist()):
                results.append(demo_predict(str(row)) if not use_model else predict(str(row), model, tokenizer))
                progress.progress((i+1)/len(df))
            progress.empty()

            out_df = df.copy()
            out_df["predicted_type"] = [r["type"] for r in results]
            out_df["confidence"]     = [f\'{r["confidence"]*100:.1f}%\' for r in results]
            st.dataframe(out_df, use_container_width=True)

            fig = px.pie(out_df["predicted_type"].value_counts().reset_index(),
                         names="predicted_type", values="count", hole=0.4)
            fig.update_layout(paper_bgcolor="#151820", font_color="#e8e4dc", height=300)
            st.plotly_chart(fig, use_container_width=True)

            st.download_button("⬇ Download Results", out_df.to_csv(index=False).encode(), "results.csv", "text/csv")
'''

# ── pages/explorer.py ─────────────────────────────────────────
explorer_code = '''
import streamlit as st
from utils.mbti_data import MBTI_TYPES

GROUPS = {
    "🔵 Analysts (NT)":  ["INTJ","INTP","ENTJ","ENTP"],
    "🟢 Diplomats (NF)": ["INFJ","INFP","ENFJ","ENFP"],
    "🟡 Sentinels (SJ)": ["ISTJ","ISFJ","ESTJ","ESFJ"],
    "🟠 Explorers (SP)": ["ISTP","ISFP","ESTP","ESFP"],
}

def render():
    st.markdown("<div style=\'font-family:DM Serif Display,serif;font-size:2.2rem;color:#c8a96e\'> MBTI Explorer</div>", unsafe_allow_html=True)
    st.markdown("<div style=\'color:#6b7280\'>Browse all 16 personality types.</div>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    for tab, (group_name, type_list) in zip(st.tabs(list(GROUPS.keys())), GROUPS.items()):
        with tab:
            for i in range(0, len(type_list), 2):
                cols = st.columns(2)
                for col, mbti_type in zip(cols, type_list[i:i+2]):
                    info  = MBTI_TYPES[mbti_type]
                    color = info["color"]
                    col.markdown(f"""
                    <div class=\'mbti-card\' style=\'border-left:3px solid {color}\'>
                        <div style=\'display:flex;align-items:center;gap:1rem\'>
                            <div style=\'font-size:2rem\'>{info["emoji"]}</div>
                            <div>
                                <div style=\'font-family:DM Serif Display,serif;font-size:1.5rem;color:{color}\'>{mbti_type}</div>
                                <div style=\'font-size:0.9rem\'>{info["name"]}</div>
                                <div style=\'font-size:0.8rem;color:#6b7280;font-style:italic\'>"{info["tagline"]}"</div>
                            </div>
                        </div>
                    </div>
                    """, unsafe_allow_html=True)
                    with col.expander(f"Details — {mbti_type}"):
                        d1,d2 = st.columns(2)
                        with d1:
                            st.markdown(f"**About:** {info[\'description\']}")
                            st.markdown("**Strengths:**")
                            for s in info["strengths"]: st.markdown(f"<span style=\'color:#4ade80\'>✓ {s}</span>", unsafe_allow_html=True)
                        with d2:
                            st.markdown("**Famous People:**")
                            for p in info["famous"]: st.markdown(f"⭐ {p}")
                            st.markdown("**Careers:**")
                            for c in info["careers"]: st.markdown(f"💼 {c}")
'''

# Write all page files
pages = {
    "/content/mbti-streamlit/pages/home.py":      home_code,
    "/content/mbti-streamlit/pages/predictor.py": predictor_page_code,
    "/content/mbti-streamlit/pages/analytics.py": analytics_code,
    "/content/mbti-streamlit/pages/batch.py":     batch_code,
    "/content/mbti-streamlit/pages/explorer.py":  explorer_code,
}

for path, code in pages.items():
    with open(path, "w") as f:
        f.write(code)
    print(f"✅ {path.split('/')[-1]} created!")

print("\n✅ All pages created!")

In [ ]:
import os

required = [
    "/content/mbti-streamlit/app.py",
    "/content/mbti-streamlit/pages/home.py",
    "/content/mbti-streamlit/pages/predictor.py",
    "/content/mbti-streamlit/pages/analytics.py",
    "/content/mbti-streamlit/pages/batch.py",
    "/content/mbti-streamlit/pages/explorer.py",
    "/content/mbti-streamlit/utils/mbti_data.py",
    "/content/mbti-streamlit/utils/predictor.py",
]

print("Checking files:")
all_ok = True
for f in required:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'} {f.split('/')[-1]}")
    if not exists:
        all_ok = False

print("\n" + ("✅ All files ready! Run the ngrok cell now." if all_ok else "❌ Some files missing — rerun the cells above."))

In [ ]:
import os, shutil, subprocess, time
from pyngrok import ngrok, conf

# ── STEP 1: Copy model files FIRST ───────────────────────────
print("📁 Copying model files into app...")

os.makedirs("/content/mbti-streamlit/model", exist_ok=True)

drive_model     = "/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras"
drive_tokenizer = "/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl"

# Verify files exist in Drive before copying
if not os.path.exists(drive_model):
    print("❌ Model not found in Drive!")
    print("   Run Cell 9 first to save your trained model.")
    raise SystemExit()

if not os.path.exists(drive_tokenizer):
    print("❌ Tokenizer not found in Drive!")
    print("   Run Cell 9 first to save your tokenizer.")
    raise SystemExit()

# Copy both files into app folder
shutil.copy(drive_model,     "/content/mbti-streamlit/model/mbti_bilstm_model.keras")
shutil.copy(drive_tokenizer, "/content/mbti-streamlit/model/tokenizer.pkl")

# Confirm copy succeeded
for f in ["/content/mbti-streamlit/model/mbti_bilstm_model.keras",
          "/content/mbti-streamlit/model/tokenizer.pkl"]:
    size = os.path.getsize(f) / (1024*1024)
    print(f"  ✅ {f.split('/')[-1]} — {size:.1f} MB")

# ── STEP 2: Kill all old processes ───────────────────────────
print("\n🔄 Killing old processes...")
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
ngrok.kill()
time.sleep(4)
print("✅ Cleared!")

# ── STEP 3: Launch Streamlit AFTER model is ready ────────────
print("\n🚀 Starting Streamlit...")
subprocess.Popen(
    ["streamlit", "run", "/content/mbti-streamlit/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.address", "0.0.0.0"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(10)  # wait for full boot
print("✅ Streamlit running!")

# ── STEP 4: Open ngrok tunnel ─────────────────────────────────
NGROK_TOKEN = "3AAWg3GY3FvZWGA2Vrx5KqznOyK_6rPEQ7wgKfAJEByYP4dzv"
conf.get_default().auth_token = NGROK_TOKEN

tunnel = ngrok.connect(8501, "http")

print("\n" + "="*55)
print("  🎉 MBTI Classifier is LIVE!")
print(f"  🌐 {tunnel.public_url}")
print("="*55)
print("  Refresh your browser tab — model is now loaded!")


# !pip install -q streamlit pyngrok plotly

# from pyngrok import ngrok, conf
# import subprocess, time, os, shutil

# NGROK_TOKEN = "3AAWg3GY3FvZWGA2Vrx5KqznOyK_6rPEQ7wgKfAJEByYP4dzv"   # ← your token
# conf.get_default().auth_token = NGROK_TOKEN

# # Copy model
# os.makedirs("/content/mbti-streamlit/model", exist_ok=True)
# shutil.copy(
#     "/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras",
#     "/content/mbti-streamlit/model/mbti_bilstm_model.keras"
# )
# shutil.copy(
#     "/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl",
#     "/content/mbti-streamlit/model/tokenizer.pkl"
# )
# print("✅ Model files copied!")

# # Kill old processes
# subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
# ngrok.kill()
# time.sleep(3)

# # Launch Streamlit
# subprocess.Popen(
#     ["streamlit", "run", "/content/mbti-streamlit/app.py",
#      "--server.port", "8501",
#      "--server.headless", "true",
#      "--server.enableCORS", "false",
#      "--server.address", "0.0.0.0"],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )
# time.sleep(10)

# # Open tunnel
# tunnel = ngrok.connect(8501, "http")
# print("=" * 55)
# print("  🚀 MBTI Classifier is LIVE!")
# print(f"  🌐 {tunnel.public_url}")
# print("=" * 55)


In [ ]:
# import os, shutil

# # ── Check where your model is saved ──────────────────────────
# print("Checking model files in Drive...")

# drive_model     = "/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras"
# drive_tokenizer = "/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl"

# print(f"  Model:     {'✅ Found' if os.path.exists(drive_model)     else '❌ Not found'}")
# print(f"  Tokenizer: {'✅ Found' if os.path.exists(drive_tokenizer) else '❌ Not found'}")

# # ── Copy into streamlit app folder ────────────────────────────
# print("\nCopying into Streamlit app...")
# os.makedirs("/content/mbti-streamlit/model", exist_ok=True)

# shutil.copy(drive_model,     "/content/mbti-streamlit/model/mbti_bilstm_model.keras")
# shutil.copy(drive_tokenizer, "/content/mbti-streamlit/model/tokenizer.pkl")

# # ── Verify ────────────────────────────────────────────────────
# print("\nVerifying app model folder:")
# for f in ["/content/mbti-streamlit/model/mbti_bilstm_model.keras",
#           "/content/mbti-streamlit/model/tokenizer.pkl"]:
#     if os.path.exists(f):
#         size = os.path.getsize(f) / (1024*1024)
#         print(f"  ✅ {f.split('/')[-1]} — {size:.1f} MB")
#     else:
#         print(f"  ❌ {f.split('/')[-1]} — still missing!")

# print("\n✅ Done! Refresh your Streamlit browser tab now.")

In [ ]:
# import pickle, os

# # Save model right now from memory
# os.makedirs("/content/drive/MyDrive/MBTI_Classifier/model", exist_ok=True)

# model.save("/content/drive/MyDrive/MBTI_Classifier/model/mbti_bilstm_model.keras")
# print("✅ Model saved!")

# with open("/content/drive/MyDrive/MBTI_Classifier/model/tokenizer.pkl", "wb") as f:
#     pickle.dump(tokenizer, f)
# print("✅ Tokenizer saved!")